In [ ]:
dbutils.widgets.dropdown(
    "run_fixture_tests",
    "false",
    ["false", "true"]
)

run_fixture_tests = (
    dbutils.widgets.get("run_fixture_tests") == "true"
)

In [ ]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "retail_operations_dev")
catalog = dbutils.widgets.get("catalog").strip()

payment_messages = (
    spark.table(f"{catalog}.bronze.payments_raw")
    .select(
        F.col("key").cast("string").alias("raw_key"),
        F.col("value").cast("string").alias("raw_json"),
    )
)

display(payment_messages)

In [ ]:
from pyspark.sql import types as T

spark.conf.set("spark.sql.session.timeZone", "UTC")

# Define the expected payment-event structure.

payment_schema = T.StructType([
    T.StructField("event_id", T.StringType(), True),
    T.StructField("event_type", T.StringType(), True),
    T.StructField("schema_version", T.IntegerType(), True),
    T.StructField("payment_id", T.StringType(), True),
    T.StructField("payment_version", T.LongType(), True),
    T.StructField("order_id", T.StringType(), True),
    T.StructField("event_time", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("amount", T.StringType(), True),
    T.StructField("currency", T.StringType(), True),
    T.StructField("payment_method", T.StringType(), True),
    T.StructField("_corrupt_record", T.StringType(), True),
])


# Parse the message while preserving its original text.

parsed_payments = (
    payment_messages
    .withColumn(
        "message_key",
        F.get_json_object("raw_key" , "$")
    )
    .withColumn(
        "payment",
        F.from_json(
            F.col("raw_json"),
            payment_schema,
            {
                "mode": "PERMISSIVE",
                "columnNameOfCorruptRecord": "_corrupt_record",
            },
        )
    )
    .select(
        "raw_key",
        "raw_json",
        "message_key",
        "payment.*",
    )
    .withColumnRenamed("amount", "amount_raw")
    .withColumn(
        "amount",
        F.expr("try_cast(amount_raw AS DECIMAL(18,2))")
    )
    .withColumn(
        "event_timestamp",
        F.try_to_timestamp("event_time")
    )
)
display(
    parsed_payments.select(
        "payment_id",
        "order_id",
        "payment_version",
        "status",
        "amount_raw",
        "amount",
        "currency",
        "event_timestamp",
        "_corrupt_record",
    )
)

In [ ]:
# Required strings must contain meaningful text.

def has_text(column):
    return F.coalesce(
        F.length(F.trim(column)) > 0,
        F.lit(False)
    )

# Each condition describes valid data.

payment_rules = [
    (
        has_text(F.col("raw_json"))
        & F.col("_corrupt_record").isNull(),
        "Malformed or missing payment JSON"
    ),
    (
        has_text(F.col("event_id")),
        "Missing event_id"
    ),
    (
        has_text(F.col("payment_id")),
        "Missing payment_id"
    ),
    (
        has_text(F.col("order_id")),
        "Missing order_id"
    ),
    (
        has_text(F.col("message_key"))
        & (F.col("message_key") == F.col("payment_id")),
        "Kafka key missing or different from payment_id"
    ),
    (
        F.col("schema_version") == 1,
        "Missing or unsupported schema_version"
    ),
    (
        F.col("payment_version") > 0,
        "Missing or invalid payment_version"
    ),
    (
        F.col("event_timestamp").isNotNull(),
        "Missing or invalid event_time"
    ),
    (
        F.col("event_type") == "payment_captured",
        "Missing or unsupported event_type"
    ),
    (
        F.col("status") == "captured",
        "Missing or unsupported payment status"
    ),
    (
        F.col("currency") == "GBP",
        "Missing or unsupported currency"
    ),
    (
        F.col("payment_method").isin(
            "card", "bank_transfer", "wallet"
        ),
        "Missing or unsupported payment_method"
    ),
    (
        F.col("amount_raw").rlike(
            r"^[0-9]+(\.[0-9]{1,2})?$"
        ),
        "Amount must contain digits with at most two decimal places"
    ),
    (
        F.col("amount").isNotNull()
        & (F.col("amount") > 0),
        "Amount must be positive and fit DECIMAL(18, 2)"
    ),
]

# Collect all failed checks.

checked_payments = (
    parsed_payments
    .withColumn(
        "rejection_reason",
        F.filter(
            F.array(*[
                F.when(
                    ~F.coalesce(condition, F.lit(False)),
                    F.lit(reason)
                )
                for condition , reason in payment_rules
            ]),
            lambda reason : reason.isNotNull()
        )
    )
    .withColumn(
        'is_valid',
        (F.size('rejection_reason') == 0)   
    )
)

In [ ]:
valid_payments = (
    checked_payments
    .filter(
        F.col("is_valid")
    )
)

quarantine_payments = (
    checked_payments
    .filter(
        ~F.col("is_valid")
    )
)

display(
    checked_payments
    .groupBy("is_valid")
    .count()
)

display(
    checked_payments.select(
        "event_id",
        "payment_id",
        "order_id",
        "amount_raw",
        "amount",
        "currency",
        "is_valid",
        "rejection_reason",
    )
)

In [ ]:
def parse_and_validate_payments(raw_df):
    parsed_df = (
        raw_df
        .select(
            F.col("key").cast("string").alias("raw_key"),
            F.col("value").cast("string").alias("raw_json"),
        )
        .withColumn(
            "message_key",
            F.get_json_object("raw_key", "$")
        )
        .withColumn(
            "payment",
            F.from_json(
                F.col("raw_json"),
                payment_schema,
                {
                    "mode": "PERMISSIVE",
                    "columnNameOfCorruptRecord": "_corrupt_record",
                },
            )
        )
        .select(
            "raw_key",
            "raw_json",
            "message_key",
            "payment.*",
        )
        .withColumnRenamed("amount", "amount_raw")
        .withColumn(
            "amount",
            F.expr("try_cast(amount_raw AS DECIMAL(18, 2))")
        )
        .withColumn(
            "event_timestamp",
            F.try_to_timestamp("event_time")
        )
    )

    return (
        parsed_df
        .withColumn(
            "rejection_reasons",
            F.filter(
                F.array(*[
                    F.when(
                        ~F.coalesce(condition, F.lit(False)),
                        F.lit(reason)
                    )
                    for condition, reason in payment_rules
                ]),
                lambda reason: reason.isNotNull()
            )
        )
        .withColumn(
            "is_valid",
            F.size("rejection_reasons") == 0
        )
    )

In [ ]:
# Ensure the storage objects exist.
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {catalog}.operations"
)

spark.sql(
    f"CREATE VOLUME IF NOT EXISTS "
    f"{catalog}.operations.streaming_checkpoints"
)

payment_source_table = f"{catalog}.bronze.payments_raw"

payment_validated_table = (
    f"{catalog}.operations.payment_events_validated"
)

payment_checkpoint_path = (
    f"/Volumes/{catalog}/operations/streaming_checkpoints/"
    "bronze_to_validated_payments_v1"
)

print("Source:", payment_source_table)
print("Destination:", payment_validated_table)
print("Checkpoint:", payment_checkpoint_path)

In [ ]:
# Define the streaming source.

raw_payment_stream = (
    spark.readStream.table(
        payment_source_table
    )
)

# Parse, validate, and attach the processing timestamp.

checked_payment_stream = (
    parse_and_validate_payments(raw_payment_stream)
    .withColumn(
        "validated_at",
        F.current_timestamp()
    )
)

# Start the incremental write.
payment_query = (
    checked_payment_stream
    .writeStream
    .format('delta')
    .option("checkpointLocation" , payment_checkpoint_path)
    .outputMode("append")
    .trigger(availableNow = True)
    .toTable(payment_validated_table)
)

payment_query.awaitTermination()

print("Payment validation run completed.")

In [ ]:
saved_payment_events = spark.table(
    payment_validated_table
)

display(
    saved_payment_events
    .groupBy("is_valid")
    .count()
)

display(
    saved_payment_events.select(
        "event_id",
        "payment_id",
        "order_id",
        "amount",
        "currency",
        "is_valid",
        "rejection_reasons",
        "validated_at",
    )
)

In [ ]:
# Read all persisted payment-validation results.

saved_payment_events = spark.table(
    payment_validated_table
)

quanrantine_payment_events = saved_payment_events.filter(
    ~F.col("is_valid")
)

# Define the business fields used to compare events.

payment_fields = [
    "event_id",
    "event_type",
    "schema_version",
    "payment_id",
    "payment_version",
    "order_id",
    "event_time",
    "status",
    "amount",
    "currency",
    "payment_method",
]

# Rebuild a consistent JSON representation of valid events.
payment_candidates = (
    saved_payment_events
    .filter(F.col('is_valid'))
    .withColumn(
        "event_payload",
        F.to_json(
            F.struct(*[
                F.col(field_name)
                for field_name in payment_fields
            ]),
            options = {"ignoreNullFields" : "false"}
        )
    )
)

In [ ]:
payment_event_checks = (
    payment_candidates
    .groupBy("event_id")
    .agg(
        F.count('*').alias("delivery_count"),
        F.countDistinct("event_payload").alias("payload_versions"),
    )
    .withColumn(
        "classification",
        F.when(
            F.col('payload_versions') > 1,
            F.lit("conflict")
        )
        .when(
            F.col('delivery_count') > 1,
            F.lit("duplicate")
        )
        .otherwise(F.lit("unique"))
    )
)

display(payment_event_checks)

In [ ]:
from pyspark.sql.window import Window

# Attach each event's classification to its full records.

classified_payments = payment_candidates.join(
    payment_event_checks,
    on = "event_id",
    how = "inner",
)

# Preserve all deliveries associated with conflicting IDs.

conflicting_payments = classified_payments.filter(
    F.col("classification") == "conflict"
)

# Rank identical deliveries within each non-conflicting event ID
payment_delivery_window = (
    Window
    .partitionBy("event_id")
    .orderBy(
        F.col("validated_at").asc(),
        F.col("raw_json").asc(),
        F.col("raw_key").asc(  )
    )
)

deduplicated_payments = (
    classified_payments
    .filter(
        F.col('classification')  != "conflict"  
    )
    .withColumn(
        "delivery_rank",
        F.row_number().over(payment_delivery_window)
    )
    .filter(
        F.col("delivery_rank") == 1
    )
    .drop("delivery_rank")
)

display(
    deduplicated_payments.select(
        "event_id",
        "payment_id",
        "payment_version",
        "order_id",
        "amount",
        "delivery_count",
        "classification",
    )
)

display(
    conflicting_payments.select(
        "event_id",
        "payment_id",
        "event_payload",
    )
)

In [ ]:
if run_fixture_tests:
    # Read current order lines for generating sample payments.
    order_items = spark.table(
        f"{catalog}.silver.order_items_current"
    )

    payment_seed = (
        order_items
        .filter(
            (F.col("order_status") != "cancelled")
            & (F.col("order_id") != "O10001")
        )
        .groupBy("order_id", "currency")
        .agg(
            F.sum("line_amount").alias("order_total")
        )
        .orderBy("order_id")
    )

    # Each order must have only one currency.
    duplicate_order_ids = (
        payment_seed
        .groupBy("order_id")
        .count()
        .filter(F.col("count") > 1)
    )

    assert duplicate_order_ids.limit(1).count() == 0, (
        "An order has multiple currencies."
    )

    # Sample payments require positive GBP order totals.
    invalid_totals = payment_seed.filter(
        F.col("order_total").isNull()
        | (F.col("order_total") <= 0)
        | F.col("currency").isNull()
        | (F.col("currency") != "GBP")
    )

    assert invalid_totals.limit(1).count() == 0, (
        "An order has an invalid total or unsupported currency."
    )

    print(
        "Orders available for payment generation:",
        payment_seed.count()
    )

    display(payment_seed)

else:
    print("Skipped sample-payment seed preparation.")

In [ ]:
# Inspect only the batch we just sent.
batch_payments = (
    spark.table(payment_validated_table)
    .filter(
        F.col("payment_id").startswith("PAY-B001-")
    )
)

display(
    batch_payments
    .groupBy("is_valid")
    .count()
    .orderBy("is_valid")
)

display(
    batch_payments
    .filter(~F.col("is_valid"))
    .select(
        "payment_id",
        "order_id",
        "amount_raw",
        "rejection_reasons",
    )
)

In [ ]:
# Stop before changing Silver if an event ID has different payloads.
if conflicting_payments.limit(1).count() > 0:
    display(conflicting_payments)

    raise ValueError(
        "Conflicting payment event IDs detected. "
        "Silver payments were not updated by this run. "
        "Review the conflicting records before retrying."
    )

from delta.tables import DeltaTable

payment_events_table = f"{catalog}.silver.payment_events"

# Verify that the merge source has one row per event ID.

duplicate_event_ids = (
    deduplicated_payments
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_event_ids.limit(1).count() > 0 :
    display(duplicate_event_ids)
    raise ValueError('Duplicate event IDs remain in the merge source.')

# Expose the prepared DataFrame to SQL.
deduplicated_payments.createOrReplaceTempView(
    "deduplicated_payment_events_source"
)

# Create an empty Delta table if needed.
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {payment_events_table}
    USING DELTA
    AS 
    SELECT *
    FROM deduplicated_payment_events_source
    WHERE 1 = 0      
          
""")

# Synchronize using the COMPLETE non-conflicting event history.

(
    DeltaTable.forName(spark , payment_events_table)
    .alias('target')
    .merge(
        deduplicated_payments.alias("source"),
        "target.event_id = source.event_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .whenNotMatchedBySourceDelete()
    .execute()
)

print(f"Saved: {payment_events_table}")

In [ ]:
saved_payment_history = spark.table(payment_events_table)

summary_df = saved_payment_history.agg(
    F.count("*").alias("saved_events"),
    F.countDistinct("event_id").alias("unique_event_ids"),
    F.countDistinct("payment_id").alias("unique_payment_ids"),
)

display(summary_df)

summary = summary_df.first()

# Always run these general checks.
assert summary["saved_events"] == summary["unique_event_ids"], (
    "Duplicate event IDs exist in Silver."
)

assert summary["saved_events"] == deduplicated_payments.count(), (
    "Saved event count differs from the prepared source."
)

# Check the original test batch only when explicitly enabled.
if run_fixture_tests:
    saved_batch = saved_payment_history.filter(
        F.col("payment_id").startswith("PAY-B001-")
    )

    assert saved_batch.count() == 31, (
        "Batch B001 should contain 31 distinct valid events."
    )

    invalid_test_payments = saved_payment_history.filter(
        F.col("payment_id").isin(
            "PAY-B001-INVALID-NEGATIVE",
            "PAY-B001-INVALID-MISSING-ORDER",
        )
    )

    assert invalid_test_payments.count() == 0, (
        "An invalid test payment reached the Silver event table."
    )

else:
    print("Skipped the B001 fixture tests.")

print("Payment-event history checks passed.")

In [ ]:
from pyspark.sql.window import Window

payment_history = spark.table(
    f"{catalog}.silver.payment_events"
)

# Different events must not claim the same payment version.
version_collisions = (
    payment_history
    .groupBy("payment_id" , "payment_version")
    .count()
    .filter(F.col("count") > 1)
)

if version_collisions.limit(1).count() > 0:
    display(version_collisions)
    raise ValueError(
        "Multiple events claim the same payment version."
    )

# A payment must retain its order and currency across versions.

identity_conflicts = (
    payment_history
    .groupBy("payment_id")
    .agg(
        F.countDistinct("order_id").alias('order_count'),
        F.countDistinct("currency").alias('currency_count'),
    )
    .filter(
        (F.col("order_count") > 1)
        | (F.col("currency_count") > 1)
    )
)

if identity_conflicts.limit(1).count() > 0:
    display(identity_conflicts)
    raise ValueError(
        "A payment changes its order or currency across versions."
    )

# Rank each payment's versions from highest to lowest.

payment_window = (
    Window
    .partitionBy("payment_id")
    .orderBy(F.col("payment_version").desc())
)

current_payments = (
    payment_history
    .withColumn(
        "version_rank",
        F.row_number().over(payment_window)
    )
    .filter(
        F.col("version_rank") == 1
    )
    .select(
        "payment_id",
        "payment_version",
        F.col("event_id").alias("latest_event_id"),
        "order_id",
        "event_timestamp",
        "status",
        "amount",
        "currency",
        "payment_method",   
    )
)

display(current_payments)

In [ ]:
from delta.tables import DeltaTable

current_payments_table = (
    f"{catalog}.silver.payments_current"
)

current_payments.createOrReplaceTempView(
    "current_payments_source"
)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {current_payments_table} 
    USING DELTA
    AS
    SELECT *
    FROM current_payments_source
    WHERE 1 = 0      
""")

(
    DeltaTable.forName(spark,current_payments_table)
    .alias('target')
    .merge(
        current_payments.alias("source"),
        "target.payment_id = source.payment_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .whenNotMatchedBySourceDelete()
    .execute()
)

print(f"Saved: {current_payments_table}")

In [ ]:
saved_current_payments = spark.table(
    current_payments_table
)

summary_df = saved_current_payments.agg(
    F.count("*").alias("saved_payments"),
    F.countDistinct("payment_id").alias("unique_payment_ids"),
)

display(summary_df)

summary = summary_df.first()

assert summary["saved_payments"] == summary["unique_payment_ids"], (
    "Duplicate payment IDs exist in payments_current."
)

assert summary["saved_payments"] == current_payments.count(), (
    "Saved payment count differs from the prepared source."
)

display(
    saved_current_payments
    .filter(F.col("order_id") == "O10001")
    .select(
        "payment_id",
        "order_id",
        "payment_version",
        "amount",
        "currency",
        "status",
    )
)

print("Current-payment checks passed.")